# 04 — Visual retrieval (E5): ColQwen2 against OCR + BM25

**Objective.** Test whether a pretrained visual page retriever puts more annotated evidence in front of GPT-4o-mini than the application's OCR + BM25 retriever. The comparison counts the pages that **actually reach the model**: the top 5 after the application's context guard (3–4 pages at 768 px), not the raw top 5.

| | |
|---|---|
| **Specification** | `experiments/specs/e5_visual_retrieval_v1.json`, frozen in git before any visual score existed (this notebook refuses to run on an edited copy) |
| **Input** | `questions.parquet`, `pages.parquet`, `render_manifest_e1.csv` and the 768-px page images (00); `rankings.parquet`, `retrieval_per_question.csv` (01); the frozen E0 text run and E4 full run (regression references); 14 held-out MMLongBench-Doc PDFs (downloaded here, CC BY-NC 4.0, git-ignored); the checkpoint `vidore/colqwen2-v1.0-hf` at a pinned revision (downloaded once, 4.4 GB, git-ignored) |
| **Output** | `data/processed/mmlongbench_doc/e5/`: held-out page records and render manifest, rankings, per-question table, summary, encoding log, figures (regenerable); embeddings in `data/cache/e5/` (versioned by model revision and configuration); frozen copy in `experiments/frozen/e5/` |
| **Paid calls** | none. Nothing in this notebook calls an LLM or VLM API. |
| **Next** | If retrieval improves, a separate, paid end-to-end experiment (E6) with the same reader and request format; otherwise, the reader (`02`, `03`). |


## Method

### 1. Why test visual retrieval now

E4 showed that the application answers 11/40 answerable questions, against 17/37 for the same reader given the annotated evidence pages. Part of the gap is page selection. BM25's top 5 holds an evidence page for 32/40 questions, but the guard keeps only 3–4 pages, and those hold one for 29/40: 8 questions are not retrieved and 3 lose their rank-5 page to the guard. In a slide deck exported as images, BM25 depends on OCR, and a question about a chart or a photo may share few words with the page. A retriever that reads the **page image** might find such evidence. It might also put the evidence higher, which is what matters once only 3–4 pages survive the guard.

### 2. The visual retriever: ColQwen2 v1.0, a ColPali-family model

**ColPali** (Faysse et al., 2024, arXiv:2407.01449) treats a page as an image. The pipeline has no OCR and no layout parsing:

- **Encoder.** A vision-language model (PaliGemma-3B: SigLIP-So400m patch encoder + Gemma-2B) encodes the page, after a 448 × 448 resize, as 1,024 patch tokens plus a short prompt. Each output token is projected to $D = 128$ dimensions and L2-normalised.
- **Index.** A page is therefore a **bag of vectors** $\{d_j\}$, not one vector.
- **Query.** A query is encoded by the same language model as text tokens $\{q_i\}$, padded with a few augmentation tokens.
- **Score (late interaction, ColBERT's MaxSim).** Each query token finds its best-matching page token, and the matches are summed:

$$s(q, d) = \sum_{i=1}^{N_q} \max_{1 \le j \le N_d} \langle q_i, d_j \rangle .$$

- **Training.** One epoch of in-batch contrastive training with LoRA, on 118,695 English query–page pairs: 63 % from academic sets (DocVQA, InfoVQA, TAT-DQA, arXivQA), 37 % synthetic questions on web-crawled PDF pages.
- **Reported results.** On the ViDoRe benchmark (nDCG@5), ColPali scores 81.3 on average. BM25 on Unstructured + OCR text scores 65.5. The gap is largest on figures and tables (ArxivQA 79.1 vs 31.6) and smallest on text-heavy reports (AI 96.2 vs 92.8, TAT-DQA 65.8 vs 62.7).

**Why it might find what BM25 misses.**

- Page tokens carry layout, charts, logos and photographs, not only recognised words.
- A token can match a paraphrase or a visual concept, not just the same spelling.
- MaxSim lets each query word find its own region of the page.

**Costs and limits.**

- **Indexing.** It needs a 2–3B-parameter model run on every page: the paper reports 0.39 s per page on an NVIDIA L4.
- **Storage.** About 1,000 vectors × 128 × 2 bytes ≈ 250 KB per page, roughly 100× a BM25 posting list.
- **Scoring.** Cost grows with pages × query tokens × page tokens.
- **Training distribution.** English, mostly single pages.
- **Resolution.** A fixed 448-px square: a dense US-letter page loses detail, and its aspect ratio is distorted.

**ColVision models** keep the recipe (same authors, same training data, same MaxSim) and change the backbone:

- **ColQwen2 / ColQwen2.5** use Qwen2(.5)-VL, which encodes the page at its native aspect ratio with a variable number of tokens.
- **ColSmol** uses SmolVLM (256M/500M) for small devices.

**Choice for this experiment: `vidore/colqwen2-v1.0-hf`, revision `ddc07d23`.** It is **ColQwen2, not ColPali**, and is named so throughout. Four reasons:

1. **Same method and data as ColPali**, so the result tests the ColPali idea.
2. **Native aspect ratio**, up to 602,112 pixels (≤ 768 visual tokens). ColPali squeezes a page into 200,704 pixels.
3. **Fits this machine.** 2.2B parameters, a 4.4 GB checkpoint, Apache-2.0 licence. ColPali v1.3 is 2.9B parameters and 5.9 GB, on an 8 GB laptop.
4. **Runs on the installed library.** transformers 5.23.0 ships `ColQwen2ForRetrieval` and `ColQwen2Processor`, so no new framework is needed.

ColPali v1.3 itself is left untested. That is a limitation, not a substitution.

### 3. Design

- **Arm A: OCR + BM25.** The application's retriever, **unchanged**: notebook 00's text, notebook 01's BM25 ($k_1 = 1.5$, $b = 0.75$).
- **Arm V: ColQwen2 v1.0.** Page images (the application's own 768-px answering images) and MaxSim, with the processor's documented defaults.
- **Arm H: rank fusion (exploratory).** Reciprocal rank fusion $1/(60 + r_A) + 1/(60 + r_V)$. It is reported, never decided on.

Every arm is built the same way:

- **Input.** It ranks **every page of the question's PDF** from **the question string alone**, and returns 1-based physical page numbers. Evidence pages, answers, answer formats and sources are never passed to a ranking function (checked in section 6).
- **Context selection.** The same rule turns each ranking into the pages the model would receive. Take the top 5, then keep the longest prefix whose prompt fits $128{,}000 - 400$ tokens. For $S$ = kept pages and $q$ = the question,

$$100 + \text{bytes}(\text{system}) + \text{bytes}(\text{Question: } q \ldots) + \sum_{p \in S} \big(\text{bytes}(\texttt{[Page } p\texttt{]}) + \text{tok}(w_p, h_p)\big) \le 127{,}600,$$

where $\text{tok}(w, h) = 2{,}833 + 5{,}667 \times$ (512-px tiles after GPT-4o-mini's resizing). Section 4 checks this rule against the 57 page lists E4 actually sent.

### 4. Metrics

For annotated evidence pages $E$, supplied pages $S$ and retrieved top-$k$ pages $R_k$:

$$\text{Recall}_{\text{after}} = \frac{|E \cap S|}{|E|}, \qquad \text{Hit}_{\text{after}} = \mathbb 1\big[|E \cap S| > 0\big], \qquad \text{Complete}_{\text{after}} = \mathbb 1\big[E \subseteq S\big].$$

**Primary endpoint:** held-out mean $\text{Recall}_{\text{after}}$. **Key secondaries:** Hit and Complete.

**Secondary measures:**

- **Before the guard:** Recall@$k$ and Hit@$k$ for $k = 1, 3, 4, 5$.
- **Ranking quality:** MRR on the full ranking, $1/\min_{p \in E} \text{rank}(p)$, and binary nDCG@5, $\sum_{i \le 5} \text{rel}_i / \log_2(i+1)$ divided by its ideal value.
- **Context guard:** evidence lost to it.
- **Cost:** time, memory and storage.

### 5. Populations, statistics, decision rule (all frozen in the spec)

**Populations.**

- **Development** (6 documents, 57 questions, 40 in the evidence population). Earlier experiments chose the application's settings on it. It is used only for regression against the frozen BM25/E4 numbers and for diagnostics.
- **Held-out** (14 documents, 118 questions). Two documents per MMLongBench-Doc type, drawn by a seeded rule from documents this repository never touched, at most 100 pages each. The rule and its output were committed before any visual score existed.

**Statistics.**

- **Unit.** Each question is compared with itself across arms: $\Delta = \overline{m_V - m_A}$.
- **Clustering.** Questions cluster in documents, so the interval is a **document-cluster bootstrap**: 10,000 resamples of the documents, 95 % percentile.
- **Tests.** An exact document-level sign-flip test, and an exact McNemar test on Hit (descriptive, since it ignores clustering).
- **Variability.** Retrieval is deterministic, which is checked. The uncertainty is therefore sampling of documents and questions, not run-to-run noise, and E4's 5/37 reader-variability margin does not apply.

**Decision rule** (held-out, primary endpoint, $\delta = 0.05$):

| verdict | condition |
|---|---|
| **visual_improves_evidence** | all four hold: $\Delta \ge 0.05$; interval lower bound $> 0$; $\Delta_{\text{Hit}} \ge 0$; questions with less evidence under V $\le$ ½ × questions with more |
| **visual_worse** | $\Delta \le -0.05$ and interval upper bound $< 0$ |
| **no_meaningful_difference** | interval inside $(-0.05, 0.05)$ |
| **inconclusive** | anything else |

**Assumptions.** Evidence annotations are single, so another page may also contain the answer, which makes all recalls pessimistic for both arms. A hit is necessary but not sufficient for a correct answer.


In [ ]:
from collections import Counter
from itertools import product
from math import ceil, comb, log, log2
from pathlib import Path
from types import SimpleNamespace
import ast
import gc
import hashlib
import json
import os
import platform
import re
import shutil
import subprocess
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil
import pymupdf
from IPython.display import Markdown, display

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 170)


def find_project_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("Could not find the repository root")


PROJECT_ROOT = find_project_root()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "mmlongbench_doc"
E5_DIR = PROCESSED_DIR / "e5"
MMLONG_DIR = PROJECT_ROOT / "data" / "external" / "mmlongbench_doc"
PDF_DIR = MMLONG_DIR / "documents"
TESSDATA_DIR = PROJECT_ROOT / "resources" / "tessdata"
IMAGE_DIR = PROJECT_ROOT / "data" / "cache" / "mmlongbench_doc" / "page_images" / "short768-long1536"  # E1's 768-px profile
SPEC_PATH = PROJECT_ROOT / "experiments" / "specs" / "e5_visual_retrieval_v1.json"
FROZEN_E0_TEXT_RUN = PROJECT_ROOT / "experiments" / "frozen" / "mmlongbench_doc" / "text_run_20261119T220858Z_a351a532"
FROZEN_E4_RUN = PROJECT_ROOT / "experiments" / "frozen" / "e4" / "run_20261123T000625940289Z_15119ec0_full"
FROZEN_E5_DIR = PROJECT_ROOT / "experiments" / "frozen" / "e5"
E5_DIR.mkdir(parents=True, exist_ok=True)

# The spec was committed before any visual score existed. Evaluation refuses an edited copy: a change is a new version.
spec_relative = str(SPEC_PATH.relative_to(PROJECT_ROOT))
git = lambda *args: subprocess.run(["git", *args], cwd=PROJECT_ROOT, capture_output=True, check=True).stdout
freeze_commits = git("log", "--format=%H", "--diff-filter=A", "--", spec_relative).decode().split()
assert freeze_commits, f"{spec_relative} is not committed: freeze the spec before evaluating"
SPEC_FREEZE_COMMIT = freeze_commits[-1]
assert git("show", f"{SPEC_FREEZE_COMMIT}:{spec_relative}") == SPEC_PATH.read_bytes(), \
    "the E5 spec differs from its frozen commit; any change is a new spec version (e5-visual-retrieval-v2)"
SPEC = json.loads(SPEC_PATH.read_text())
assert SPEC["spec_id"] == "e5-visual-retrieval-v1" and SPEC["status"] == "frozen-before-evaluation"

SETTINGS = {
    "dataset_repo": "yubo2333/MMLongBench-Doc",
    "dataset_revision": "2ff6aa9237fc777b6627dc57a486e9225ac5fb86",
    "ocr_min_native_words": 30, "ocr_dpi": 150,                     # notebook 00
    "bm25_k1": 1.5, "bm25_b": 0.75, "top_k": 5,                    # notebook 01 / the application
    "image_short_px": 768, "image_long_px": 1536,                  # E1 profile = the application's answering images
    "heldout_seed": 20261123, "heldout_per_type": 2, "heldout_max_pages": 100, "heldout_max_mb": 15.0,
    "spot_check_n": 8,
    "k_values": [1, 3, 4, 5], "rrf_k": 60,
    "bootstrap_resamples": 10_000, "stat_seed": 20261123,
    "min_meaningful_difference": SPEC["hypotheses"]["minimum_meaningful_difference"],
}
assert SETTINGS["dataset_revision"] in SPEC["populations"]["held_out"]["source"]
assert str(SETTINGS["heldout_seed"]) in SPEC["populations"]["held_out"]["selection_rule"]
print(f"Spec {SPEC['spec_id']} frozen in commit {SPEC_FREEZE_COMMIT[:7]} at {SPEC['frozen_at']}; the file on disk equals it.")
print("Paid API calls in this notebook:", SPEC["paid_api_calls"])


## 1. Environment and resource feasibility

Before loading 2.2 billion parameters on a laptop, the cell below checks what the machine has and estimates what the model needs. It uses the **safetensors header** (tensor names and shapes, read without loading any weights) and the token counts of the pre-freeze probe.

**Measured before the spec was frozen** (three development pages, recorded in the spec):

- The full model in bf16 on MPS holds about 5.5 GB of GPU memory on this 8 GB M1 and runs at **35–45 s per page**, with the system swapping. Queries take 0.2 s. The cost is all in the 675M-parameter vision tower, which reads ~3,000 image patches.
- Running the two halves **in two passes** removes the memory contention and gives **~9.5 s (vision) + ~3.7 s (language model) per page**. Pass 1 runs only the vision tower over every page and caches its output. Pass 2 runs the library's own forward with the vision tower replaced by a stub that returns the cached output.
- The two passes do **identical arithmetic**: their embeddings were bitwise equal to the single-pass forward on all three pages. Section 5 checks this again.


In [ ]:
import torch
import transformers
from safetensors import safe_open
from safetensors.torch import load_file, save_file

MODEL = SPEC["arms"]["V_colqwen2"]["checkpoint"]
MODEL_DIR = PROJECT_ROOT / "data" / "cache" / "hf" / f"{MODEL['repo'].split('/')[-1]}@{MODEL['revision'][:12]}"
WEIGHTS = MODEL_DIR / MODEL["weights_file"]


def sha256_file(path, chunk=1 << 24):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        while block := handle.read(chunk):
            digest.update(block)
    return digest.hexdigest()


memory, disk = psutil.virtual_memory(), shutil.disk_usage(PROJECT_ROOT)
cpu = subprocess.run(["sysctl", "-n", "machdep.cpu.brand_string"], capture_output=True, text=True).stdout.strip() if sys.platform == "darwin" else platform.processor()
ENVIRONMENT = {
    "platform": f"{platform.system()} {platform.mac_ver()[0] or platform.release()} {platform.machine()}", "cpu": cpu,
    "cpu_cores": psutil.cpu_count(), "ram_gb": round(memory.total / 1e9, 1), "ram_available_gb_now": round(memory.available / 1e9, 1),
    "disk_free_gb_now": round(disk.free / 1e9, 1), "python": sys.version.split()[0], "torch": torch.__version__,
    "transformers": transformers.__version__, "pymupdf": pymupdf.VersionBind, "mps_available": torch.backends.mps.is_available(),
    "cuda_available": torch.cuda.is_available(),
}
display(pd.Series(ENVIRONMENT, name="value").to_frame())
assert transformers.__version__ == "5.23.0" and torch.__version__.startswith("2.15.0"), "versions differ from the spec: a new condition"
assert ENVIRONMENT["mps_available"], "the spec fixes the Apple MPS device; no fallback is pre-authorised (see the spec)"

# Download the pinned checkpoint once (4.4 GB, free, git-ignored), then verify its bytes against the spec.
if not WEIGHTS.is_file():
    assert disk.free > 8e9, f"downloading the checkpoint needs about 4.5 GB; only {disk.free / 1e9:.1f} GB free"
    from huggingface_hub import snapshot_download
    snapshot_download(MODEL["repo"], revision=MODEL["revision"], local_dir=MODEL_DIR)
weights_check_path = E5_DIR / "weights_check.json"
stat = WEIGHTS.stat()
check = json.loads(weights_check_path.read_text()) if weights_check_path.is_file() else {}
if check.get("size") != stat.st_size or check.get("mtime_ns") != stat.st_mtime_ns:  # hash once per file version (~1 min)
    check = {"size": stat.st_size, "mtime_ns": stat.st_mtime_ns, "sha256": sha256_file(WEIGHTS)}
    weights_check_path.write_text(json.dumps(check))
assert check["size"] == MODEL["weights_bytes"] and check["sha256"] == MODEL["weights_sha256"], "checkpoint bytes differ from the spec"

# What each pass must hold in memory, from the safetensors header (no weights loaded).
with safe_open(WEIGHTS, framework="pt") as weights:
    tensor_sizes = {name: int(np.prod(weights.get_slice(name).get_shape())) for name in weights.keys()}
vision_params = sum(n for name, n in tensor_sizes.items() if ".visual." in name)
total_params = sum(tensor_sizes.values())
assert total_params == MODEL["parameters"]
n_dev_pages, n_heldout_pages = 195, SPEC["populations"]["held_out"]["size_at_selection"]["pages"]
tokens_per_page = 760  # 731-779 tokens on the three probe pages (<= 768 visual tokens + prompt)
estimates = pd.Series({
    "parameters, total": f"{total_params / 1e9:.2f} B ({total_params * 2 / 1e9:.2f} GB in bf16)",
    "pass 1: vision tower": f"{vision_params / 1e6:.0f} M parameters, {vision_params * 2 / 1e9:.2f} GB",
    "pass 2: language model + projection": f"{(total_params - vision_params) / 1e9:.2f} B parameters, {(total_params - vision_params) * 2 / 1e9:.2f} GB",
    "pages to encode (dev + held-out)": f"{n_dev_pages} + {n_heldout_pages} = {n_dev_pages + n_heldout_pages}",
    "embedding storage": f"~{tokens_per_page} x 128 x 2 bytes = {tokens_per_page * 256 / 1e3:.0f} KB per page, ~{(n_dev_pages + n_heldout_pages) * tokens_per_page * 256 / 1e6:.0f} MB in all",
    "temporary vision features": f"~744 x 1536 x 2 bytes = 2.3 MB per page, ~{(n_dev_pages + n_heldout_pages) * 2.3 / 1e3:.1f} GB until pass 2 consumes them",
    "time (probe: ~13 s per page)": f"~{(n_dev_pages + n_heldout_pages) * 13 / 3600:.1f} h for an empty cache; minutes when cached",
}, name="estimate").to_frame()
display(estimates)
assert disk.free > 3e9 or (E5_DIR / "encoding_log.csv").is_file(), "need ~3 GB free disk for temporary features and embeddings"
print(f"Checkpoint verified: {WEIGHTS.relative_to(PROJECT_ROOT)}, {check['size']:,} bytes, sha256 {check['sha256'][:12]}… = spec.")


## 2. Dataset preparation and validation

**Development set.** It is read from notebooks 00 and 01 without change: 6 documents, 57 questions, and notebook 01's evidence population of 40 answerable questions.

**Held-out set.** The spec's rule is re-executed here, and its output must equal the document list committed in the spec. The rule:

- Exclude the 6 development documents and the four other MMLongBench-Doc documents named anywhere in this repository.
- Keep documents of at most 15 MB.
- For each of the 7 document types, in sorted order, permute the type's eligible documents with one shared seeded generator.
- Accept documents in that order if they have 1–100 pages, until 2 are accepted per type.

The rule reads only document metadata (type, size, page count), never a question, answer or retrieval result.

**Evidence population** (same definition for both sets): answerable questions with at least one evidence page, no page 0 and every evidence page inside the PDF. Rows excluded by each condition are counted.


In [ ]:
for name in ("questions.parquet", "pages.parquet", "rankings.parquet", "retrieval_per_question.csv", "render_manifest_e1.csv"):
    assert (PROCESSED_DIR / name).is_file(), f"{name} is missing: run notebooks 00 and 01 first"
for path in (FROZEN_E0_TEXT_RUN / "scored_predictions.csv", FROZEN_E4_RUN / "execution_log.jsonl"):
    assert path.is_file(), f"{path.relative_to(PROJECT_ROOT)} is missing: restore experiments/frozen/ from git"

bench = pd.read_parquet(PROCESSED_DIR / "questions.parquet")  # the full benchmark table at the pinned revision (00)
for column in ("evidence_pages", "evidence_sources"):
    bench[column] = bench[column].map(list)
assert len(bench) == 1091 and bench["qid"].is_unique


def evidence_population(questions, page_counts):
    # The retrieval population of notebook 01, with the reason each other row is excluded.
    reasons = pd.DataFrame({
        "unanswerable": ~questions["answerable"],
        "no evidence page": questions["answerable"] & (questions["n_evidence_pages"] == 0),
        "cites page 0": questions["evidence_pages"].map(lambda e: 0 in e),
        "page beyond the PDF": [max(e, default=0) > page_counts[d] for e, d in zip(questions["evidence_pages"], questions["doc_id"])],
    }, index=questions.index)
    return questions[~reasons.any(axis=1)].copy(), reasons.sum().astype(int)


dev_q = bench[bench["in_subset"]].reset_index(drop=True)
dev_pages = pd.read_parquet(PROCESSED_DIR / "pages.parquet")
DEV_DOCS = sorted(dev_q["doc_id"].unique())
dev_page_count = dev_pages.groupby("doc_id")["page"].max()
dev_pop, dev_excluded = evidence_population(dev_q, dev_page_count)
assert len(dev_q) == 57 and len(dev_pages) == 195 and len(DEV_DOCS) == 6
assert set(dev_pop["qid"]) == set(pd.read_csv(PROCESSED_DIR / "retrieval_per_question.csv")["qid"]) and len(dev_pop) == 40
print(f"Development: {len(dev_q)} questions, {len(dev_pages)} pages, {len(DEV_DOCS)} documents; evidence population "
      f"{len(dev_pop)} (= notebook 01). Excluded rows: {dev_excluded[dev_excluded > 0].to_dict()}")


In [ ]:
from huggingface_hub import HfApi, hf_hub_download

HELD_OUT_SPEC = SPEC["populations"]["held_out"]
OTHER_TOUCHED = ["05-03-18-political-release.pdf", "2312.04350v3.pdf", "PH_2016.06.08_Economy-Final.pdf", "RAR.pdf"]
assert all(name in " ".join(HELD_OUT_SPEC["exclusions"]) for name in OTHER_TOUCHED)
EXCLUDED_DOCS = set(DEV_DOCS) | set(OTHER_TOUCHED)

hub_files = HfApi().list_repo_tree(SETTINGS["dataset_repo"], path_in_repo="documents", repo_type="dataset", revision=SETTINGS["dataset_revision"])
hub_mb = {Path(item.path).name: item.size / 1e6 for item in hub_files if item.path.endswith(".pdf")}
doc_type = bench.groupby("doc_id")["doc_type"].first()
assert set(doc_type.index) <= set(hub_mb)


def download_pdf(doc_id):
    return Path(hf_hub_download(SETTINGS["dataset_repo"], f"documents/{doc_id}", repo_type="dataset",
                                revision=SETTINGS["dataset_revision"], local_dir=MMLONG_DIR))


def pdf_page_count(path):
    with pymupdf.open(path) as pdf:
        return pdf.page_count


rng = np.random.default_rng(SETTINGS["heldout_seed"])  # one generator, shared across the types in sorted order
selection_log = []
for kind in sorted(doc_type.unique()):
    eligible = sorted(d for d in doc_type[doc_type == kind].index if d not in EXCLUDED_DOCS and hub_mb[d] <= SETTINGS["heldout_max_mb"])
    accepted = 0
    for doc_id in (eligible[i] for i in rng.permutation(len(eligible))):
        n_pages = pdf_page_count(download_pdf(doc_id))
        ok = 1 <= n_pages <= SETTINGS["heldout_max_pages"]
        selection_log.append({"doc_type": kind, "doc_id": doc_id, "eligible_of_type": len(eligible), "pages": n_pages,
                              "size_mb": round(hub_mb[doc_id], 2), "accepted": ok})
        accepted += ok
        if accepted == SETTINGS["heldout_per_type"]:
            break
selection_log = pd.DataFrame(selection_log)
HELDOUT_DOCS = selection_log.loc[selection_log["accepted"], "doc_id"].tolist()

# The re-executed rule must reproduce the list committed in the spec, including the rejected documents.
assert HELDOUT_DOCS == HELD_OUT_SPEC["selected_documents"], "the selection rule no longer reproduces the frozen list"
assert selection_log.loc[~selection_log["accepted"]].set_index("doc_id")["pages"].to_dict() == HELD_OUT_SPEC["rejected_by_page_cap"]
assert not set(HELDOUT_DOCS) & EXCLUDED_DOCS
print(f"Selection reproduced: {len(HELDOUT_DOCS)} documents accepted of {len(selection_log)} examined "
      f"(= the spec, including the 3 rejected by the page cap); none of the {len(EXCLUDED_DOCS)} excluded documents.")
selection_log.assign(doc_id=selection_log["doc_id"].str.slice(0, 48))


In [ ]:
held_pdf = {doc_id: download_pdf(doc_id) for doc_id in HELDOUT_DOCS}
held_page_count = pd.Series({doc_id: pdf_page_count(path) for doc_id, path in held_pdf.items()})
held_q = bench[bench["doc_id"].isin(HELDOUT_DOCS)].reset_index(drop=True)
held_pop, held_excluded = evidence_population(held_q, held_page_count)
size = HELD_OUT_SPEC["size_at_selection"]
assert (len(HELDOUT_DOCS), int(held_page_count.sum()), len(held_q), int(held_q["answerable"].sum())) == \
       (size["documents"], size["pages"], size["questions"], size["answerable"])
assert not held_q["in_subset"].any()

held_docs_table = pd.DataFrame({
    "type": doc_type[HELDOUT_DOCS], "pages": held_page_count, "size_mb": pd.Series(hub_mb)[HELDOUT_DOCS].round(2),
    "questions": held_q.groupby("doc_id").size(), "answerable": held_q.groupby("doc_id")["answerable"].sum(),
    "evidence population": held_pop.groupby("doc_id").size(),
    "multi-page": held_pop[held_pop["n_evidence_pages"] > 1].groupby("doc_id").size(),
}).fillna(0).astype({"evidence population": int, "multi-page": int})
held_docs_table.index = held_docs_table.index.str.slice(0, 44)
print(f"Held-out: {len(held_q)} questions ({int(held_q['answerable'].sum())} answerable) in {len(HELDOUT_DOCS)} documents, "
      f"{int(held_page_count.sum())} pages; evidence population {len(held_pop)} "
      f"({int((held_pop['n_evidence_pages'] > 1).sum())} multi-page). Excluded rows: {held_excluded[held_excluded > 0].to_dict()}")
display(held_docs_table)


def multi_page_share(questions):
    with_evidence = questions["answerable"] & (questions["n_evidence_pages"] > 0)
    return round(100 * (with_evidence & (questions["n_evidence_pages"] > 1)).sum() / with_evidence.sum(), 1)


comparison = pd.concat({
    "benchmark %": bench["answer_format"].value_counts(normalize=True), "development %": dev_q["answer_format"].value_counts(normalize=True),
    "held-out %": held_q["answer_format"].value_counts(normalize=True)}, axis=1).fillna(0).mul(100).round(1)
comparison.loc["multi-page, % of answerable with evidence"] = [multi_page_share(f) for f in (bench, dev_q, held_q)]
comparison


### Held-out page records: the baseline's text and the application's images

Both representations are produced by the **existing** pipeline, rebuilt here line for line.

- **Text, for BM25 (notebook 00's policy).** Take the native text layer (`get_text("text", sort=True)`). If it has fewer than 30 words, replace it with full-page Tesseract OCR at 150 dpi.
- **Image, for ColQwen2 and the context guard.** Fit the whole page in a 768 × 1536-px box with zoom $z = \min(768/\min(w, h),\ 1536/\max(w, h))$ and save it as an opaque RGB PNG. This is E1's profile, the image the application sends to GPT-4o-mini.

Text is cached in `e5/heldout_pages.parquet`, with the settings that produced it: OCR is the slow step. Images are re-rendered on every run and must equal the cached bytes, which checks that rendering is deterministic. Section 4 checks both against the application's own functions.


In [ ]:
assert (TESSDATA_DIR / "eng.traineddata").is_file(), "OCR model missing: resources/tessdata/eng.traineddata"


def page_text_record(page):
    # Notebook 00's extraction rule, unchanged: native text, or full-page OCR below the word threshold.
    native_text = page.get_text("text", sort=True)
    native_words = len(native_text.split())
    ocr_used = native_words < SETTINGS["ocr_min_native_words"]
    if ocr_used:
        ocr_textpage = page.get_textpage_ocr(language="eng", dpi=SETTINGS["ocr_dpi"], full=True, tessdata=str(TESSDATA_DIR))
        text = page.get_text("text", textpage=ocr_textpage, sort=True)
    else:
        text = native_text
    return {"native_words": native_words, "ocr_used": ocr_used, "text": text, "words": len(text.split()),
            "width_pt": page.rect.width, "height_pt": page.rect.height, "rotation": page.rotation}


text_cache, text_meta_path = E5_DIR / "heldout_pages.parquet", E5_DIR / "heldout_pages.settings.json"
text_meta = {"documents": HELDOUT_DOCS, "ocr_min_native_words": SETTINGS["ocr_min_native_words"], "ocr_dpi": SETTINGS["ocr_dpi"],
             "pymupdf": pymupdf.VersionBind, "tessdata_sha256": sha256_file(TESSDATA_DIR / "eng.traineddata")}
cached_meta = json.loads(text_meta_path.read_text()) if text_meta_path.is_file() else {}
if text_cache.is_file() and {k: cached_meta.get(k) for k in text_meta} == text_meta:
    held_pages = pd.read_parquet(text_cache)
    text_seconds = cached_meta["seconds"]
    print(f"Loaded {len(held_pages)} held-out page records from the cache (extraction took {text_seconds:.0f} s when computed).")
else:
    started, rows = time.perf_counter(), []
    for doc_id in HELDOUT_DOCS:
        with pymupdf.open(held_pdf[doc_id]) as pdf:
            for index, page in enumerate(pdf):
                rows.append({"doc_id": doc_id, "page": index + 1, **page_text_record(page)})
    held_pages = pd.DataFrame(rows)
    text_seconds = time.perf_counter() - started
    held_pages.to_parquet(text_cache, index=False)
    text_meta_path.write_text(json.dumps({**text_meta, "seconds": text_seconds}))
    print(f"Extracted {len(held_pages)} held-out pages in {text_seconds:.0f} s; OCR on {int(held_pages['ocr_used'].sum())} pages.")
assert len(held_pages) == int(held_page_count.sum())
held_pages.groupby("doc_id").agg(pages=("page", "size"), ocr_pages=("ocr_used", "sum"), median_words=("words", "median"),
                                 page_pt=("width_pt", lambda w: "/".join(sorted({f"{x:.0f}x{y:.0f}" for x, y in zip(w, held_pages.loc[w.index, "height_pt"])})))
                                 ).rename(index=lambda d: d[:44])


In [ ]:
def render_answering_png(page):
    # The application's answering image (E1 profile): whole page, rotation applied, opaque RGB, fitted in 768 x 1536 px.
    zoom = min(SETTINGS["image_short_px"] / min(page.rect.width, page.rect.height),
               SETTINGS["image_long_px"] / max(page.rect.width, page.rect.height))
    return page.get_pixmap(matrix=pymupdf.Matrix(zoom, zoom), alpha=False).tobytes("png")


def image_path(doc_id, page_number):
    return IMAGE_DIR / Path(doc_id).stem / f"page_{page_number:03d}.png"


started, rows, rewritten = time.perf_counter(), [], 0
for doc_id in HELDOUT_DOCS:
    with pymupdf.open(held_pdf[doc_id]) as pdf:
        for index, page in enumerate(pdf):
            png, target = render_answering_png(page), image_path(doc_id, index + 1)
            if not target.is_file() or target.read_bytes() != png:
                rewritten += target.is_file()  # an existing file with different bytes would mean non-deterministic rendering
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_bytes(png)
            width, height = pymupdf.Pixmap(png).width, pymupdf.Pixmap(png).height
            rows.append({"doc_id": doc_id, "page": index + 1, "width_px": width, "height_px": height,
                         "sha256": hashlib.sha256(png).hexdigest(), "image_path": str(target.relative_to(PROJECT_ROOT))})
held_manifest = pd.DataFrame(rows)
held_manifest.to_csv(E5_DIR / "render_manifest_heldout.csv", index=False)
assert rewritten == 0, f"{rewritten} cached images differed from a fresh render"
print(f"Rendered {len(held_manifest)} held-out pages in {time.perf_counter() - started:.0f} s; every cached image equals a fresh render.")

# Development images were rendered by notebook 00 (E1 profile); verify their bytes instead of re-rendering.
dev_manifest = pd.read_csv(PROCESSED_DIR / "render_manifest_e1.csv")
assert len(dev_manifest) == 195 and all(hashlib.sha256((PROJECT_ROOT / p).read_bytes()).hexdigest() == h
                                        for p, h in zip(dev_manifest["image_path"], dev_manifest["sha256"]))

# One table of every page both retrievers search: population, 1-based page, text for BM25, image for ColQwen2 and the guard.
all_pages = pd.concat([
    dev_manifest[["doc_id", "page", "width_px", "height_px", "sha256", "image_path"]]
    .merge(dev_pages[["doc_id", "page", "text", "ocr_used"]], on=["doc_id", "page"], validate="one_to_one").assign(population="development"),
    held_manifest.merge(held_pages[["doc_id", "page", "text", "ocr_used"]], on=["doc_id", "page"], validate="one_to_one").assign(population="held-out"),
], ignore_index=True)
assert not all_pages.duplicated(["doc_id", "page"]).any()
assert all(sorted(group["page"]) == list(range(1, len(group) + 1)) for _, group in all_pages.groupby("doc_id"))
PAGE_SIZE = {(d, p): (w, h) for d, p, w, h in zip(all_pages["doc_id"], all_pages["page"], all_pages["width_px"], all_pages["height_px"])}
print(f"{len(all_pages)} pages in {all_pages['doc_id'].nunique()} documents; image sizes (w x h) by population:")
all_pages.groupby("population").apply(lambda f: (f["width_px"].astype(str) + "x" + f["height_px"].astype(str)).value_counts().head(6).to_dict())


### Annotation quality, page alignment and licence

The held-out annotations come from the same benchmark, revision and protocol as the development set. Notebook 00 verified the 1-based physical page convention there: a synthetic PDF, OCR of real pages, and the answer strings. Here the convention is checked in three ways:

1. **Mechanical.** Every evidence page lies inside its PDF, and anomalies are counted.
2. **Retriever-based** (in section 4, as in notebook 01). BM25 never sees the annotations. If the pages were shifted by one, its top-1 page would sit next to the annotated page more often than on it.
3. **By eye.** A seeded sample of 8 answerable questions, each shown with its reference answer and evidence pages. The labels below were recorded by viewing these pages **before any visual-retrieval score was computed**. They describe annotation quality only and never filter questions.

**Licence.** MMLongBench-Doc data are CC BY-NC 4.0, for research use. The PDFs stay git-ignored; no page image or question text is committed.


In [ ]:
annotation_checks = pd.DataFrame({
    population: {
        "questions": len(q), "answerable": int(q["answerable"].sum()), "evidence population": len(pop),
        "multi-page questions in population": int((pop["n_evidence_pages"] > 1).sum()),
        "evidence pages in population": int(pop["evidence_pages"].map(lambda e: len(set(e))).sum()),
        "unanswerable rows listing pages": int((~q["answerable"] & (q["n_evidence_pages"] > 0)).sum()),
        **{f"excluded: {reason}": int(n) for reason, n in excluded.items() if reason != "unanswerable"},
        "duplicate page in one annotation": int(q["evidence_pages"].map(lambda e: len(e) != len(set(e))).sum()),
    }
    for population, q, pop, excluded in (("development", dev_q, dev_pop, dev_excluded), ("held-out", held_q, held_pop, held_excluded))
})
display(annotation_checks)
held_sources = held_pop.explode("evidence_sources")["evidence_sources"].value_counts()
print("Held-out evidence sources (a question with several counts once per source):", held_sources.to_dict())
# Evidence pages of the population are 1-based physical pages inside their PDF (the convention verified in notebook 00).
assert all(1 <= p <= held_page_count[d] for d, e in zip(held_pop["doc_id"], held_pop["evidence_pages"]) for p in e)


In [ ]:
spot_rng = np.random.default_rng(SETTINGS["heldout_seed"])  # a fresh generator, as the spec states
spot_qids = sorted(spot_rng.choice(sorted(held_pop["qid"]), size=SETTINGS["spot_check_n"], replace=False).tolist())
spot = held_pop.set_index("qid").loc[spot_qids]

# Recorded by viewing each evidence page (the images below) against the reference answer, before any retrieval score.
SPOT_CHECK_LABELS = {}  # qid -> (label, note); label in {"supported", "partly", "not supported"}

fig, axes = plt.subplots(2, 4, figsize=(13, 8.5))
for ax, (qid, row) in zip(axes.flat, spot.iterrows()):
    page = row["evidence_pages"][0]
    ax.imshow(plt.imread(PROJECT_ROOT / all_pages.set_index(["doc_id", "page"]).loc[(row["doc_id"], page), "image_path"]))
    ax.set_title(f"{qid} · p.{page}" + (f" (+{len(row['evidence_pages']) - 1} more)" if len(row["evidence_pages"]) > 1 else ""), fontsize=8)
    ax.axis("off")
plt.suptitle("Spot check: first evidence page of 8 seeded held-out questions", fontsize=10)
plt.tight_layout()
plt.show()
pd.DataFrame({
    "doc": spot["doc_id"].str.slice(0, 26), "question": spot["question"].str.slice(0, 80), "reference": spot["answer"].astype(str).str.slice(0, 40),
    "evidence": spot["evidence_pages"], "sources": spot["evidence_sources"],
    "label": [SPOT_CHECK_LABELS.get(q, ("(not yet labelled)", ""))[0] for q in spot_qids],
    "note": [SPOT_CHECK_LABELS.get(q, ("", ""))[1] for q in spot_qids],
}, index=spot_qids)


**Spot check.** (Filled in after viewing; see the labels above.)


## 3. BM25 baseline (arm A), reproduced

Notebook 01's BM25 is rebuilt here, with nothing changed:

- **Tokens:** lower-cased runs of `[a-z0-9]`.
- **Statistics:** computed per document, $k_1 = 1.5$, $b = 0.75$.
- **Ties:** broken by page number.

It must reproduce the frozen E0 run's **full** ranking for all 57 development questions, and notebook 01's Recall@5 0.689 and Hit@5 0.800.

**Retrieval input.** Both arms rank from one table with three columns: `qid`, `doc_id` and `question`. It carries no evidence pages, answers, formats or sources.


In [ ]:
TOKEN_PATTERN = re.compile(r"[a-z0-9]+")


def tokenize(text):
    return TOKEN_PATTERN.findall(str(text).lower())


def build_bm25_index(page_texts):
    # page_texts: {page_number: text} for one PDF -> the statistics BM25 needs (notebook 01).
    tokens = {page: tokenize(text) for page, text in page_texts.items()}
    frequency = Counter(token for page_tokens in tokens.values() for token in set(page_tokens))
    n_pages = len(tokens)
    return {"term_counts": {page: Counter(t) for page, t in tokens.items()}, "lengths": {page: len(t) for page, t in tokens.items()},
            "avgdl": float(np.mean([len(t) for t in tokens.values()])),
            "idf": {token: log(1 + (n_pages - n_t + 0.5) / (n_t + 0.5)) for token, n_t in frequency.items()}}


def bm25_rank(index, query, k1=SETTINGS["bm25_k1"], b=SETTINGS["bm25_b"]):
    # [(page, score), ...] for every page, best first; ties broken by page number.
    terms = set(tokenize(query))
    scored = []
    for page, counts in index["term_counts"].items():
        length_factor = k1 * (1 - b + b * index["lengths"][page] / max(index["avgdl"], 1.0))
        scored.append((page, sum(index["idf"][t] * counts[t] * (k1 + 1) / (counts[t] + length_factor) for t in terms if counts[t])))
    return sorted(scored, key=lambda item: (-item[1], item[0]))


# The only input either retriever receives: question id, document and question string.
retrieval_questions = pd.concat([dev_q.assign(population="development"), held_q.assign(population="held-out")], ignore_index=True)[
    ["qid", "population", "doc_id", "question"]]
assert retrieval_questions["qid"].is_unique and len(retrieval_questions) == 57 + 118

started = time.perf_counter()
bm25_indexes = {doc_id: build_bm25_index(dict(zip(group["page"], group["text"]))) for doc_id, group in all_pages.groupby("doc_id")}
bm25_index_seconds = time.perf_counter() - started
started = time.perf_counter()
RANKINGS = {"A": {row.qid: bm25_rank(bm25_indexes[row.doc_id], row.question) for row in retrieval_questions.itertuples()}}
bm25_query_seconds = (time.perf_counter() - started) / len(retrieval_questions)

# Regression: full development rankings equal the frozen E0 text run (every page, every position); 01's numbers hold.
frozen_e0 = pd.read_csv(FROZEN_E0_TEXT_RUN / "scored_predictions.csv").set_index("qid")["ranking"].map(ast.literal_eval)
assert all([page for page, _ in RANKINGS["A"][qid]] == frozen_e0[qid] for qid in dev_q["qid"]) and len(frozen_e0) == 57
top5 = {qid: {page for page, _ in RANKINGS["A"][qid][:5]} for qid in dev_pop["qid"]}
dev_recall5 = np.mean([len(set(e) & top5[q]) / len(set(e)) for q, e in zip(dev_pop["qid"], dev_pop["evidence_pages"])])
dev_hit5 = np.mean([bool(set(e) & top5[q]) for q, e in zip(dev_pop["qid"], dev_pop["evidence_pages"])])
assert round(dev_recall5, 3) == 0.689 and round(dev_hit5, 3) == 0.800
print(f"BM25 reproduced: full rankings = frozen E0 on 57/57 development questions; Recall@5 {dev_recall5:.3f}, Hit@5 {dev_hit5:.3f} "
      f"(notebook 01: 0.689, 0.800). Index {bm25_index_seconds:.2f} s for {len(all_pages)} pages; {1e3 * bm25_query_seconds:.1f} ms per query.")


### The context guard, rebuilt

The application keeps the **longest ranked prefix** of the top 5 whose prompt-token upper bound, plus 400 output tokens, fits GPT-4o-mini's 128,000-token context.

- **Text** is bounded by its UTF-8 byte count: byte-level BPE never needs more tokens than bytes.
- **Each page image** costs what the API bills for a high-detail image:
  1. fit the image inside 2048 × 2048;
  2. shrink its short side to 768 px if it is larger;
  3. count 512-px tiles $t$;
  4. the cost is $2{,}833 + 5{,}667\,t$ tokens.

A 768 × 994 page has 2 × 2 tiles and costs 25,501 tokens; a 1366 × 768 slide has 3 × 2 tiles and costs 36,835. Four letter pages fit and five do not, so the guard is not a choice of $k$: it depends on the **page shapes** of the ranked pages.

The rebuilt guard must give exactly the page lists that E4 sent to the model for the 57 development questions. It must also reproduce E4's evidence availability: 29 of 40 questions.


In [ ]:
# The application's reader configuration (prompt mmlb-image-v1, verified by its frozen hash) and token rule.
IMAGE_PROMPT_VERSION = "mmlb-image-v1"
IMAGE_SYSTEM_PROMPT = """You answer questions about a single PDF document using only the page images provided.
Each page image is preceded by a tag [Page N], where N is the 1-based page number in the PDF.

Return a JSON object with exactly these keys:
  "answer": the shortest complete answer. Use a number (integer or decimal) when the answer is a number,
            and a JSON list when several items are asked for. If the page images do not contain the answer,
            use exactly the string "Not answerable".
  "cited_pages": a list of the page numbers N whose images support the answer ([] if "Not answerable").

Do not use knowledge outside the page images. Do not guess."""
assert hashlib.sha256((IMAGE_PROMPT_VERSION + IMAGE_SYSTEM_PROMPT).encode()).hexdigest()[:12] == "35ff2e8b91f9"
CONTEXT_TOKENS, OUTPUT_TOKENS, FRAMING_TOKENS = 128_000, 400, 100


def image_tokens(width, height):
    # GPT-4o-mini high detail: fit in 2048 x 2048, shrink the short side to 768 if larger, 2,833 + 5,667 per 512-px tile.
    scale = min(1.0, 2048 / max(width, height))
    width, height = width * scale, height * scale
    short_scale = 768 / min(width, height)
    if short_scale < 1:
        width, height = width * short_scale, height * short_scale
    return 2_833 + 5_667 * ceil(round(width, 6) / 512) * ceil(round(height, 6) / 512)


def guard(question, ranked_pages, doc_id, limit=CONTEXT_TOKENS - OUTPUT_TOKENS):
    # Longest ranked prefix whose prompt bound fits; returns (kept pages, dropped pages). Never reorders.
    total = FRAMING_TOKENS + len(IMAGE_SYSTEM_PROMPT.encode()) + len(f"Question: {question}\n\nPage images:".encode())
    kept = []
    for page in ranked_pages:
        cost = len(f"[Page {page}]".encode()) + image_tokens(*PAGE_SIZE[(doc_id, page)])
        if total + cost > limit:
            break
        kept.append(page)
        total += cost
    return kept, list(ranked_pages[len(kept):])


assert image_tokens(768, 994) == 25_501 and image_tokens(1366, 768) == 36_835 and image_tokens(1024, 768) == 25_501

# Regression against what E4 actually sent: the APP page lists and image sizes in the frozen execution log.
e4_app = {r["qid"]: r for r in map(json.loads, open(FROZEN_E4_RUN / "execution_log.jsonl")) if r["arm"] == "APP" and r["replicate_id"] == "main_r1"}
question_text = retrieval_questions.set_index("qid")["question"]
doc_of = retrieval_questions.set_index("qid")["doc_id"]
dev_supplied_A = {qid: guard(question_text[qid], [p for p, _ in RANKINGS["A"][qid][:5]], doc_of[qid])[0] for qid in dev_q["qid"]}
assert len(e4_app) == 57 and all(dev_supplied_A[qid] == e4_app[qid]["supplied_pages"] for qid in dev_q["qid"])
assert all([list(PAGE_SIZE[(doc_of[q], p)]) for p in r["supplied_pages"]] == r["image_dimensions"] for q, r in e4_app.items())
dev_hit_after_A = sum(bool(set(e) & set(dev_supplied_A[q])) for q, e in zip(dev_pop["qid"], dev_pop["evidence_pages"]))
assert dev_hit_after_A == 29
print(f"Guard reproduced: supplied pages = the frozen E4 APP requests on 57/57 questions (and their image sizes); "
      f"evidence supplied for {dev_hit_after_A}/40 answerable questions, as in E4.")


In [ ]:
# Held-out page convention, checked with the retriever (notebook 01's test): BM25 never sees the annotations, so if the
# benchmark's pages were shifted, its top-1 page would land next to the annotated page more often than on it.
offset_counts = pd.Series({
    label: int(sum(RANKINGS["A"][q][0][0] - offset in set(e) for q, e in zip(held_pop["qid"], held_pop["evidence_pages"])))
    for offset, label in {-1: "top-1 == p - 1", 0: "top-1 == p (1-based)", 1: "top-1 == p + 1 (0-based)"}.items()
}, name=f"held-out questions (of {len(held_pop)})")
assert offset_counts["top-1 == p (1-based)"] > max(offset_counts["top-1 == p - 1"], offset_counts["top-1 == p + 1 (0-based)"])
offset_counts.to_frame()


### Arm A is the application's retriever: parity on the held-out documents

On the development set, arm A equals the frozen runs (above). The held-out documents are new, so this cell compares the notebook's construction with the application's own functions on every held-out page and question:

- `visualdoc.documents.extract_pages` gives the same text;
- `visualdoc.retrieval.BM25Index.rank` gives the same full rankings and scores;
- `visualdoc.documents.render_page_png` gives the same image bytes;
- `visualdoc.answering.select_pages_within_context` keeps the same pages.

*This is the only cell that imports `src/visualdoc/`. It does so only to check that arm A, as rebuilt above, **is** the application's retriever (the AGENTS.md exception for measuring the application). No other step uses it.*


In [ ]:
APP_SRC = PROJECT_ROOT / "src"
if str(APP_SRC) not in sys.path:
    sys.path.insert(0, str(APP_SRC))  # this checkout, explicitly: macOS can hide the editable install's .pth file
from visualdoc import answering as app_answering, documents as app_documents, retrieval as app_retrieval

assert Path(app_documents.__file__).resolve().is_relative_to(APP_SRC.resolve())
APP_MODULE_SHA256 = {f"src/visualdoc/{name}.py": sha256_file(APP_SRC / "visualdoc" / f"{name}.py") for name in ("answering", "documents", "retrieval")}
assert (app_retrieval.BM25_K1, app_retrieval.BM25_B, app_retrieval.TOP_K) == (SETTINGS["bm25_k1"], SETTINGS["bm25_b"], SETTINGS["top_k"])
assert (app_documents.OCR_MIN_NATIVE_WORDS, app_documents.OCR_DPI) == (SETTINGS["ocr_min_native_words"], SETTINGS["ocr_dpi"])
assert app_answering.IMAGE_SYSTEM_PROMPT == IMAGE_SYSTEM_PROMPT and app_answering.MAX_OUTPUT_TOKENS == OUTPUT_TOKENS

started, parity = time.perf_counter(), Counter()
held_text = held_pages.set_index(["doc_id", "page"])["text"]
held_sha = held_manifest.set_index(["doc_id", "page"])["sha256"]
for doc_id in HELDOUT_DOCS:
    app_pages = app_documents.extract_pages(held_pdf[doc_id], tessdata=TESSDATA_DIR)
    parity["text pages equal"] += sum(p.text == held_text[(doc_id, p.page)] for p in app_pages)
    pngs = app_documents.render_pages(held_pdf[doc_id], [p.page for p in app_pages])
    parity["image bytes equal"] += sum(hashlib.sha256(png).hexdigest() == held_sha[(doc_id, n)] for n, png in pngs.items())
    index = app_retrieval.BM25Index({p.page: p.text for p in app_pages})
    for row in retrieval_questions[retrieval_questions["doc_id"] == doc_id].itertuples():
        app_ranking = index.rank(row.question)
        parity["rankings equal (pages and scores)"] += app_ranking == RANKINGS["A"][row.qid]
        top = [page for page, _ in app_ranking[:SETTINGS["top_k"]]]
        kept, dropped = app_answering.select_pages_within_context(row.question, [(n, pngs[n]) for n in top])
        parity["guard equal"] += ([n for n, _ in kept], dropped) == guard(row.question, top, doc_id)
expected = {"text pages equal": len(held_pages), "image bytes equal": len(held_pages),
            "rankings equal (pages and scores)": len(held_q), "guard equal": len(held_q)}
assert dict(parity) == expected, (dict(parity), expected)
print(f"Arm A = the application on the held-out set ({time.perf_counter() - started:.0f} s): {dict(parity)}")


## 4. Visual model (arm V): loading and page encoding

**Configuration.** Everything comes from the spec and from the checkpoint's own files:

- transformers' native `ColQwen2ForRetrieval` and `ColQwen2Processor`;
- bf16 on the M1 GPU (MPS), sdpa attention, batch size 1, under `torch.inference_mode()`;
- the preprocessing in the pinned `preprocessor_config.json`.

**What the processor does to a page.**

1. Qwen2-VL resizes the image so that both sides are multiples of 28 and the area lies between 3,136 and 602,112 pixels. A 768 × 994 page becomes 672 × 868.
2. It cuts the image into 14-px patches: here 48 × 62 = 2,976 patches.
3. The vision tower encodes the patches and merges each 2 × 2 block into one token: 744 visual tokens.
4. The language model reads them inside the prompt "Describe the image.", and every output token becomes a 128-d unit vector.

A query is `"Query: " + question` followed by 10 `<|endoftext|>` augmentation tokens.

**Two passes, identical arithmetic.** The library's forward computes `vlm.visual(pixel_values, grid_thw).pooler_output`, writes it into the token sequence and runs the language model. Here the forward is split in two:

- **Pass 1** loads the model, keeps only `vlm.visual` (1.3 GB) on the GPU, and stores each page's `pooler_output`.
- **Pass 2** loads the model again, replaces `vlm.visual` with a stub that returns the stored output, and calls the library's **unmodified** forward. The stub checks that the feature count matches the page's grid.

The last cell of this section re-encodes pages in a single pass and requires **bitwise** equality.

**Caching.**

- **Cache directory:** `data/cache/e5/colqwen2-v1.0-hf@ddc07d2317c8/<config hash>/`. The hash covers the model revision, weight hash, library versions, device, dtype, preprocessing, prompts and embedding rule, so a change in any of them starts a new cache.
- **Keys:** each page is keyed by the SHA-256 of its PNG, each query by the SHA-256 of its text.
- **Format:** bf16 safetensors with padding rows removed. Converting bf16 to float32 for scoring is exact.
- **Temporary files:** pass-1 features are deleted once pass 2 has used them.
- **Restarting:** an interrupted run resumes where it stopped.

In [ ]:
from PIL import Image
from transformers import ColQwen2ForRetrieval, ColQwen2Processor

V_SPEC = SPEC["arms"]["V_colqwen2"]
DEVICE, DTYPE, ATTENTION = "mps", torch.bfloat16, "sdpa"
processor = ColQwen2Processor.from_pretrained(MODEL_DIR)
preprocessor_config = json.loads((MODEL_DIR / "preprocessor_config.json").read_text())
assert (preprocessor_config["min_pixels"], preprocessor_config["max_pixels"], preprocessor_config["patch_size"], preprocessor_config["merge_size"]) == (3136, 602112, 14, 2)
assert all(str(v) in V_SPEC["page_input"] for v in (3136, 602112))
assert processor.query_prefix == "Query: " and processor.query_augmentation_token == "<|endoftext|>"
assert processor.visual_prompt_prefix.replace("\n", "\\n") in V_SPEC["visual_prompt"]

ENCODER_CONFIG = {
    "repo": MODEL["repo"], "revision": MODEL["revision"], "weights_sha256": MODEL["weights_sha256"],
    "transformers": transformers.__version__, "torch": torch.__version__, "device": DEVICE, "dtype": "bfloat16",
    "attn_implementation": ATTENTION, "batch_size": 1, "page_image": "E1 profile: 768 x 1536 box, RGB PNG",
    "preprocessor_config_sha256": sha256_file(MODEL_DIR / "preprocessor_config.json"),
    "tokenizer_sha256": sha256_file(MODEL_DIR / "tokenizer.json"),
    "visual_prompt": processor.visual_prompt_prefix, "query_prefix": processor.query_prefix,
    "query_augmentation": f"10 x {processor.query_augmentation_token}",
    "embedding_rule": "model(**batch).embeddings; rows with attention_mask 0 removed; stored bf16",
}
CONFIG_HASH = hashlib.sha256(json.dumps(ENCODER_CONFIG, sort_keys=True).encode()).hexdigest()[:12]
CACHE_DIR = PROJECT_ROOT / "data" / "cache" / "e5" / MODEL_DIR.name / CONFIG_HASH
for sub in ("pages", "queries", "vision_features"):
    (CACHE_DIR / sub).mkdir(parents=True, exist_ok=True)
(CACHE_DIR / "config.json").write_text(json.dumps(ENCODER_CONFIG, indent=1, sort_keys=True))
ENCODING_LOG = E5_DIR / "encoding_log.csv"


def page_cache_path(png_sha256):
    return CACHE_DIR / "pages" / f"{png_sha256}.safetensors"


def feature_path(png_sha256):
    return CACHE_DIR / "vision_features" / f"{png_sha256}.safetensors"


def query_key(question):
    return hashlib.sha256(question.encode()).hexdigest()


def query_cache_path(question):
    return CACHE_DIR / "queries" / f"{query_key(question)}.safetensors"


def load_page_image(relative_path):
    with Image.open(PROJECT_ROOT / relative_path) as image:
        return image.convert("RGB")


def append_log(rows):
    pd.DataFrame(rows).to_csv(ENCODING_LOG, mode="a", header=not ENCODING_LOG.is_file(), index=False)


page_jobs = all_pages.drop_duplicates("sha256")[["population", "doc_id", "page", "sha256", "image_path"]].reset_index(drop=True)
pending_pages = page_jobs[~page_jobs["sha256"].map(lambda s: page_cache_path(s).is_file())]
pending_questions = sorted({q for q in retrieval_questions["question"] if not query_cache_path(q).is_file()})
SESSION = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
print(f"Encoder configuration hash {CONFIG_HASH}; cache {CACHE_DIR.relative_to(PROJECT_ROOT)}")
print(f"{len(page_jobs)} distinct page images ({len(all_pages)} pages); {len(pending_pages)} to encode; "
      f"{retrieval_questions['question'].nunique()} distinct questions, {len(pending_questions)} to encode.")

In [ ]:
# What the model receives: one page of each shape, and one query, through the processor (no model needed).
demo_rows = all_pages.assign(shape=all_pages["width_px"].astype(str) + "x" + all_pages["height_px"].astype(str)).drop_duplicates("shape").head(4)
image_pad_id = processor.tokenizer.convert_tokens_to_ids("<|image_pad|>")
demo = []
for row in demo_rows.itertuples():
    batch = processor(images=[load_page_image(row.image_path)], return_tensors="pt")
    t, h, w = batch["image_grid_thw"][0].tolist()
    demo.append({"page": f"{row.doc_id[:24]} p.{row.page}", "answering image (px)": row.shape, "model input (px)": f"{w * 14}x{h * 14}",
                 "pixels": w * h * 196, "14-px patches": t * h * w, "visual tokens (2x2 merged)": t * h * w // 4,
                 "sequence tokens": int(batch["input_ids"].shape[1]), "image tokens in sequence": int((batch["input_ids"] == image_pad_id).sum())})
demo = pd.DataFrame(demo)
assert (demo["visual tokens (2x2 merged)"] == demo["image tokens in sequence"]).all() and (demo["pixels"] <= 602_112).all()
display(demo)
example_question = question_text["mmlb-0033"]
tokens = processor.tokenizer.convert_ids_to_tokens(processor(text=[example_question], return_tensors="pt")["input_ids"][0])
print(f"Query for mmlb-0033 ({len(tokens)} tokens): {example_question!r}")
print(tokens)
assert tokens[-11:-1] == ["<|endoftext|>"] * 10

In [ ]:
# Pass 1: the vision tower alone, on every page whose embedding is not cached yet.
def unpadded_pixel_values(batch):
    # The library's own unpadding (ColQwen2ForRetrieval.forward): keep each image's h * w patch rows.
    grid = batch["image_grid_thw"]
    offsets = grid[:, 1] * grid[:, 2]
    mask = torch.arange(batch["pixel_values"].shape[1]).unsqueeze(0) < offsets.unsqueeze(1)
    return batch["pixel_values"][mask]


process = psutil.Process()
PASS_PEAKS = {}
need_vision = pending_pages[~pending_pages["sha256"].map(lambda s: feature_path(s).is_file())]
if len(need_vision):
    started = time.perf_counter()
    model = ColQwen2ForRetrieval.from_pretrained(MODEL_DIR, dtype=DTYPE, attn_implementation=ATTENTION).eval()
    visual = model.vlm.visual
    model.vlm.visual = None
    del model
    gc.collect()
    visual.to(DEVICE)
    load_seconds = time.perf_counter() - started
    peak_rss, peak_mps, log_rows = process.memory_info().rss, torch.mps.driver_allocated_memory(), []
    with torch.inference_mode():
        for i, row in enumerate(need_vision.itertuples(), 1):
            t0 = time.perf_counter()
            batch = processor(images=[load_page_image(row.image_path)], return_tensors="pt")
            features = visual(unpadded_pixel_values(batch).to(DEVICE), grid_thw=batch["image_grid_thw"].to(DEVICE), return_dict=True).pooler_output
            torch.mps.synchronize()
            save_file({"features": features.cpu().contiguous()}, feature_path(row.sha256))
            peak_rss, peak_mps = max(peak_rss, process.memory_info().rss), max(peak_mps, torch.mps.driver_allocated_memory())
            log_rows.append({"session": SESSION, "pass": "vision", "key": row.sha256, "doc_id": row.doc_id, "page": row.page,
                             "seconds": time.perf_counter() - t0, "tokens": int(features.shape[0]), "load_seconds": load_seconds})
            if i % 50 == 0 or i == len(need_vision):
                append_log(log_rows)
                log_rows = []
                print(f"  pass 1: {i}/{len(need_vision)} pages, {time.perf_counter() - started:.0f} s", flush=True)
    PASS_PEAKS["vision"] = {"rss_gb": peak_rss / 1e9, "mps_driver_gb": peak_mps / 1e9, "load_seconds": load_seconds}
    del visual
    gc.collect()
    torch.mps.empty_cache()
    print(f"Pass 1 done: {len(need_vision)} pages in {time.perf_counter() - started:.0f} s; peak RSS {peak_rss / 1e9:.2f} GB, "
          f"peak MPS driver memory {peak_mps / 1e9:.2f} GB.")
else:
    print("Pass 1: nothing to do (every page has a cached embedding or cached vision features).")

In [ ]:
# Pass 2: the library's unmodified forward, with the vision tower replaced by the features of pass 1; then the queries.
class CachedVisionFeatures(torch.nn.Module):
    """Stands in for model.vlm.visual: returns the pass-1 output of the page being encoded."""

    def __init__(self):
        super().__init__()
        self.features = None

    def forward(self, pixel_values, grid_thw=None, return_dict=True, **kwargs):
        expected = int((grid_thw.prod(-1) // 4).sum())  # one feature per 2 x 2 block of patches
        assert self.features is not None and self.features.shape[0] == expected, "cached features do not match this page"
        return SimpleNamespace(pooler_output=self.features)


def encode_with(model, batch):
    embeddings = model(**batch).embeddings
    torch.mps.synchronize()
    keep = batch["attention_mask"][0].bool()
    return embeddings[0][keep].cpu().contiguous()  # bf16, padding rows removed


if len(pending_pages) or pending_questions:
    started = time.perf_counter()
    model = ColQwen2ForRetrieval.from_pretrained(MODEL_DIR, dtype=DTYPE, attn_implementation=ATTENTION).eval()
    stub = CachedVisionFeatures()
    model.vlm.visual = stub  # the real vision weights are released before the move to the GPU
    gc.collect()
    model.to(DEVICE)
    load_seconds = time.perf_counter() - started
    peak_rss, peak_mps, log_rows = process.memory_info().rss, torch.mps.driver_allocated_memory(), []
    with torch.inference_mode():
        for i, row in enumerate(pending_pages.itertuples(), 1):
            t0 = time.perf_counter()
            batch = processor(images=[load_page_image(row.image_path)], return_tensors="pt").to(DEVICE)
            stub.features = load_file(feature_path(row.sha256))["features"].to(DEVICE)
            embedding = encode_with(model, batch)
            save_file({"embedding": embedding}, page_cache_path(row.sha256))
            feature_path(row.sha256).unlink()
            peak_rss, peak_mps = max(peak_rss, process.memory_info().rss), max(peak_mps, torch.mps.driver_allocated_memory())
            log_rows.append({"session": SESSION, "pass": "language", "key": row.sha256, "doc_id": row.doc_id, "page": row.page,
                             "seconds": time.perf_counter() - t0, "tokens": int(embedding.shape[0]), "load_seconds": load_seconds})
            if i % 50 == 0 or i == len(pending_pages):
                append_log(log_rows)
                log_rows = []
                print(f"  pass 2: {i}/{len(pending_pages)} pages, {time.perf_counter() - started:.0f} s", flush=True)
        stub.features = None
        for question in pending_questions:
            t0 = time.perf_counter()
            embedding = encode_with(model, processor(text=[question], return_tensors="pt").to(DEVICE))
            save_file({"embedding": embedding}, query_cache_path(question))
            log_rows.append({"session": SESSION, "pass": "query", "key": query_key(question), "doc_id": "", "page": 0,
                             "seconds": time.perf_counter() - t0, "tokens": int(embedding.shape[0]), "load_seconds": load_seconds})
        if log_rows:
            append_log(log_rows)
    PASS_PEAKS["language"] = {"rss_gb": peak_rss / 1e9, "mps_driver_gb": peak_mps / 1e9, "load_seconds": load_seconds}
    del model, stub
    gc.collect()
    torch.mps.empty_cache()
    print(f"Pass 2 done: {len(pending_pages)} pages and {len(pending_questions)} queries in {time.perf_counter() - started:.0f} s; "
          f"peak RSS {peak_rss / 1e9:.2f} GB, peak MPS driver memory {peak_mps / 1e9:.2f} GB.")
else:
    print("Pass 2: nothing to do (every page and question has a cached embedding).")
assert all(page_cache_path(s).is_file() for s in page_jobs["sha256"]) and all(query_cache_path(q).is_file() for q in retrieval_questions["question"])
assert not any((CACHE_DIR / "vision_features").iterdir()), "unconsumed pass-1 features remain"

In [ ]:
# Equivalence and determinism: re-encode one page per population and two queries with the ordinary single-pass forward
# (the whole model on the GPU, a few minutes on this machine) and require bitwise equality with the two-pass cache.
# Set E5_SKIP_MODEL_CHECKS=1 to skip on a machine without the checkpoint; the result is then marked as not re-verified.
SINGLE_PASS_CHECK = {"run": os.environ.get("E5_SKIP_MODEL_CHECKS") != "1"}
if SINGLE_PASS_CHECK["run"]:
    check_pages = page_jobs.groupby("population").head(1)
    check_questions = retrieval_questions.groupby("population")["question"].first().tolist()
    model = ColQwen2ForRetrieval.from_pretrained(MODEL_DIR, dtype=DTYPE, attn_implementation=ATTENTION).eval().to(DEVICE)
    results = []
    with torch.inference_mode():
        for row in check_pages.itertuples():
            t0 = time.perf_counter()
            fresh = encode_with(model, processor(images=[load_page_image(row.image_path)], return_tensors="pt").to(DEVICE))
            results.append({"item": f"page {row.doc_id[:24]} p.{row.page}", "seconds (single pass)": round(time.perf_counter() - t0, 1),
                            "bitwise equal to cache": torch.equal(fresh, load_file(page_cache_path(row.sha256))["embedding"])})
        for question in check_questions:
            fresh = encode_with(model, processor(text=[question], return_tensors="pt").to(DEVICE))
            results.append({"item": f"query {question[:40]!r}", "seconds (single pass)": None,
                            "bitwise equal to cache": torch.equal(fresh, load_file(query_cache_path(question))["embedding"])})
    del model
    gc.collect()
    torch.mps.empty_cache()
    SINGLE_PASS_CHECK["results"] = results
    display(pd.DataFrame(results))
    assert all(r["bitwise equal to cache"] for r in results), "two-pass and single-pass embeddings differ"
    print("Single-pass forward reproduces the cached two-pass embeddings bit for bit.")
else:
    print("Skipped (E5_SKIP_MODEL_CHECKS=1): equivalence not re-verified in this execution.")

<!-- e5-score-md -->

In [ ]:
# e5-maxsim-demo

In [ ]:
# e5-rank

In [ ]:
# e5-leakage

<!-- e5-context-md -->

In [ ]:
# e5-context

<!-- e5-metrics-md -->

In [ ]:
# e5-metrics

In [ ]:
# e5-dev-results

In [ ]:
# e5-heldout-results

In [ ]:
# e5-paired

In [ ]:
# e5-decision

In [ ]:
# e5-figure

In [ ]:
# e5-breakdown

<!-- e5-results-md -->

<!-- e5-errors-md -->

In [ ]:
# e5-errors

In [ ]:
# e5-examples

<!-- e5-errors-interp -->

<!-- e5-cost-md -->

In [ ]:
# e5-cost

<!-- e5-save-md -->

In [ ]:
# e5-save

In [ ]:
# e5-frozen

<!-- e5-validation-md -->

In [ ]:
# e5-validation

<!-- e5-interpretation -->